# Exploring the ScanWatch data

## 01. Introduction

This notebook explores daily Withings ScanWatch summaries covering steps, physical activity, heart rate and sleep for 12 participants over 290 days (Matias & Wac, 2022; CC BY 4.0).

## 02. Load the dataset

In [ ]:
import pandas as pd
import plotly.express as px

from wearable_data_explorer.data import METRICS, load_all

df = load_all("../data/raw")
metric_names = [m.name for m in METRICS]
key_metrics = ["steps", "hr_day_avg", "sleep_hours"]

## 03. Dataset overview

In [ ]:
print(df.shape)
df.head()

(3480, 18)


,participant,date,season,steps,distance_km,soft_activity_min,moderate_activity_min,intense_activity_min,hr_day_avg,hr_day_min,hr_day_max,sleep_hours,time_to_sleep_min,hr_sleep_avg,hr_sleep_min,hr_sleep_max,wakeups,wake_time_min
0,1487,2021-07-06,Summer,2584.0,1.841,280.016667,1.000000,0.0,72.0,61.0,114.0,6.933333,2.0,64.0,60.0,71.0,0.0,3.0
1,1487,2021-07-07,Summer,4594.0,3.201,325.000000,10.000000,0.0,72.0,61.0,111.0,6.816667,2.0,63.0,59.0,70.0,0.0,2.0
2,1487,2021-07-08,Summer,9602.0,7.011,304.033333,73.983333,0.0,78.0,58.0,148.0,7.133333,2.0,60.0,55.0,66.0,0.0,3.0
3,1487,2021-07-09,Summer,14121.0,11.017,316.033333,70.983333,25.0,76.0,60.0,110.0,6.716667,2.0,60.0,54.0,68.0,1.0,7.0
4,1487,2021-07-10,Summer,7022.0,4.835,345.983333,11.016667,0.0,74.0,53.0,113.0,7.533333,8.0,63.0,59.0,71.0,0.0,14.0


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3480 entries, 0 to 3479
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   participant            3480 non-null   str           
 1   date                   3480 non-null   datetime64[us]
 2   season                 3480 non-null   str           
 3   steps                  3440 non-null   float64       
 4   distance_km            3440 non-null   float64       
 5   soft_activity_min      3440 non-null   float64       
 6   moderate_activity_min  3440 non-null   float64       
 7   intense_activity_min   3440 non-null   float64       
 8   hr_day_avg             3330 non-null   float64       
 9   hr_day_min             3330 non-null   float64       
 10  hr_day_max             3330 non-null   float64       
 11  sleep_hours            3239 non-null   float64       
 12  time_to_sleep_min      3239 non-null   float64       
 13  hr_sleep_avg  

## 04. Coverage and missing values

In [ ]:
# Every date lies between the overall first and last date. So if a participant has as many
# distinct dates as there are days in that period, they have each day exactly once.
days = pd.date_range(df["date"].min(), df["date"].max())
every_day_once = df.groupby("participant")["date"].nunique().eq(len(days)).all()

print(f"Participants: {df['participant'].nunique()}")
print(f"Study period: {days[0].date()} to {days[-1].date()} ({len(days)} days)")
print(f"Duplicate participant-days: {df.duplicated(['participant', 'date']).sum()}")
print(f"Every participant has every day exactly once: {every_day_once}")

Participants: 12
Study period: 2021-07-06 to 2022-04-21 (290 days)
Duplicate participant-days: 0
Every participant has every day exactly once: True


In [ ]:
# The study does not cover whole calendar months at the start and end
df.groupby(df["date"].dt.to_period("M"))["date"].nunique().rename("days in data")

date
2021-07    26
2021-08    31
2021-09    30
2021-10    31
2021-11    30
2021-12    31
2022-01    31
2022-02    28
2022-03    31
2022-04    21
Freq: M, Name: days in data, dtype: int64

Each participant has one row for every date in the 290-day study period, with no duplicate participant-days. Missing data therefore occurs within existing rows rather than through gaps in the date coverage. July 2021 and April 2022 cover only part of the month, which matters when comparing monthly averages.

In [ ]:
# Share of days with no value, in percent
df[key_metrics].isna().mean().mul(100).round(1)

steps          1.1
hr_day_avg     4.3
sleep_hours    6.9
dtype: float64

In [ ]:
# The same per participant (share, not percent)
df[key_metrics].isna().groupby(df["participant"]).mean().round(3)

,steps,hr_day_avg,sleep_hours
participant,,,
1487,0.000,0.024,0.014
2201,0.021,0.090,0.166
2210,0.034,0.079,0.093
3379,0.014,0.031,0.017
4891,0.000,0.028,0.024
5359,0.003,0.003,0.014
5544,0.021,0.024,0.203
6008,0.007,0.031,0.031
6777,0.017,0.097,0.069


In [ ]:
# One row per participant, one column per day
# 1 = no sleep value that day
sleep_missing = df.pivot(index="participant", columns="date", values="sleep_hours").isna()
px.imshow(
    sleep_missing.astype(int),
    aspect="auto",
    labels={"x": "Date", "y": "Participant", "color": "Missing"},
    title="Days without sleep data",
)

Missingness varies across participants. Sleep data is missing for 20.3% of recorded days for participant 5544, and for 16.6% and 15.9% for participants 2201 and 8176, respectively. For the remaining participants, sleep missingness is below 10%.

The dataset documentation describes missing values as random, but the observed differences between participants do not independently establish whether that assumption holds.

In [ ]:
# People have different amounts of missing data. Does it matter whether we average over all
# recorded days or give each participant equal weight?
pd.DataFrame(
    {
        "all recorded days": df[key_metrics].mean(),
        "equal weight per participant": df.groupby("participant")[key_metrics].mean().mean(),
    }
).round(2)

,all recorded days,equal weight per participant
steps,7728.53,7713.94
hr_day_avg,77.70,77.86
sleep_hours,7.57,7.59


For these three metrics, the overall means are similar whether all recorded days receive equal weight or each participant receives equal weight. The differences are small in this dataset, although this comparison does not establish that weighting is unimportant for every metric or analysis.

## 05. Metric distributions and data quality

In [ ]:
df[metric_names].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
steps,3440.0,7728.53,4531.95,511.00,4569.75,6897.50,10156.25,42818.00
distance_km,3440.0,6.40,3.99,0.00,3.61,5.54,8.48,42.05
soft_activity_min,3440.0,204.48,94.89,0.00,134.00,188.00,261.03,669.03
moderate_activity_min,3440.0,43.07,44.68,0.00,8.01,32.00,60.00,366.00
intense_activity_min,3440.0,16.43,42.48,0.00,0.00,0.00,19.00,1213.37
hr_day_avg,3330.0,77.70,9.29,39.00,71.00,77.00,84.00,135.00
hr_day_min,3330.0,50.15,7.49,9.00,45.00,49.00,54.00,135.00
hr_day_max,3330.0,142.46,33.08,45.00,116.00,140.00,172.00,212.00
sleep_hours,3239.0,7.57,1.81,0.55,6.53,7.42,8.47,22.97
time_to_sleep_min,3239.0,2.80,4.00,0.00,2.00,2.00,2.00,117.00


Some metrics contain extreme values. Daily steps reach 42,818, compared with a 75th percentile of 10,156, while recorded sleep reaches 22.97 hours, compared with a 75th percentile of 8.47 hours. These values can influence the mean, so the median may provide a useful complementary summary.

In [ ]:
active_minutes = df["soft_activity_min"] + df["moderate_activity_min"] + df["intense_activity_min"]
heart_rate_contradiction = (df["hr_day_min"] > df["hr_day_avg"]) | (
    df["hr_day_avg"] > df["hr_day_max"]
)

pd.Series(
    {
        "Nights longer than 12 h": (df["sleep_hours"] > 12).sum(),
        "Nights shorter than 3 h": (df["sleep_hours"] < 3).sum(),
        "Days where the three activity levels add up to more than 24 h": (
            active_minutes > 1440
        ).sum(),
        "Days where min, average and max heart rate contradict each other": (
            heart_rate_contradiction.sum()
        ),
        "Nights with exactly 2 min to fall asleep": (df["time_to_sleep_min"] == 2).sum(),
    },
    name="count",
)

Nights longer than 12 h                                               65
Nights shorter than 3 h                                                8
Days where the three activity levels add up to more than 24 h          2
Days where min, average and max heart rate contradict each other       3
Nights with exactly 2 min to fall asleep                            2651
Name: count, dtype: int64

In [ ]:
# Who are the nights longer than 12 hours?
df.loc[df["sleep_hours"] > 12, "participant"].value_counts()

participant
5544    49
6008     5
5359     4
2201     2
3379     2
1487     1
2210     1
9775     1
Name: count, dtype: int64

In [ ]:
# Is it a few odd nights, or a pattern? Participant 5544, nights over 12 h per month
p5544 = df[df["participant"] == "5544"]
n_long = (p5544["sleep_hours"] > 12).sum()
print(f"{n_long} of {p5544['sleep_hours'].notna().sum()} recorded nights")
(p5544["sleep_hours"] > 12).groupby(p5544["date"].dt.to_period("M")).sum()

49 of 231 recorded nights


date
2021-07    0
2021-08    2
2021-09    4
2021-10    3
2021-11    9
2021-12    8
2022-01    8
2022-02    4
2022-03    7
2022-04    4
Freq: M, Name: sleep_hours, dtype: int64

Of the 65 nights longer than 12 hours, 49 belong to participant 5544. For this participant, such records occur in every month from August 2021 to April 2022, suggesting a recurring pattern rather than a single isolated observation. The dataset alone cannot establish whether these values reflect actual sleep duration or a measurement or reporting issue.

Exactly two minutes is recorded as the time to fall asleep on 2,651 nights. This concentration is notable, but the available documentation does not establish why it occurs. Similarly, activity totals above 24 hours and inconsistent heart-rate summaries are flagged for inspection rather than automatically corrected.

## 06. Differences between participants

In [ ]:
px.box(df, x="participant", y="steps", title="Daily steps per participant").update_xaxes(
    type="category"
)

In [ ]:
px.box(df, x="participant", y="sleep_hours", title="Nightly sleep per participant").update_xaxes(
    type="category"
)

The box plots show substantial variation between participants in daily steps and nightly sleep. Pooled averages can conceal these individual differences, which supports allowing results to be grouped by participant. These comparisons describe only the 12 participants in this dataset and should not be interpreted as representative of a wider population.

## 07. Findings and implications for the app

### Main findings

- **Date coverage is complete, but metric coverage is not.** Every participant has a row for all 290 dates, although some measurements are missing. Missingness also varies between participants.
- **Some values warrant caution.** In particular, participant 5544 has repeated records of sleep exceeding 12 hours. Other checks identify unusually high activity totals and inconsistent heart-rate summaries. These records remain unchanged because their cause has not been established.
- **Averages are similar under the two weighting approaches examined.** For steps, daily average heart rate and sleep, weighting all recorded days equally produces similar means to giving each participant equal weight. This result is specific to these metrics and this dataset.

### Remaining uncertainties

- Whether the missing values are random, as described in the dataset documentation.
- Whether unusual sleep, activity and heart-rate values reflect device behaviour, reporting conventions or errors.
- How far these findings generalise beyond 12 participants and 290 days.

### Implications for the Wearable Data Explorer

- **Report the sample size.** Results should include *n*, the number of recorded observations used in the calculation.
- **Handle missing values explicitly.** Missing measurements should not be treated as zero, and averages should be described as based on recorded observations.
- **Account for partial months.** July 2021 and April 2022 do not cover full calendar months, which matters for monthly comparisons.
- **Preserve the original data.** Unusual values are not automatically removed or corrected. The median can be offered alongside the mean when extreme values may affect interpretation.
- **Constrain the analysis.** The app should validate requested metrics, groupings and aggregations against an explicit set of supported options, rather than relying on the language model alone. It provides descriptive summaries, not medical advice.